# PharmaOpt | Data Quality & Historical Shortage Audit

**Research notebook · Independent student project · October 2026**  
**Purpose:** evaluate the provenance, coverage, completeness, consistency, and labeling risks in two FDA drug-shortage source snapshots **before** making any changes to the existing prediction model.

### Questions this notebook answers
1. What does each source actually contain, and when was it captured?
2. How do initial posting dates, NDC identifiers, and status values vary?
3. How does a **December 2023 archived snapshot** compare with a **September 2026 openFDA download**?
4. What should be verified before using these records to label 2022–2025 package-year observations?

**Important guardrails:** The archive is a point-in-time list, not a complete event-history database. The 2026 download is also a later snapshot. Differences between snapshots **do not prove that one is complete or that every missing record was an actual shortage**. A `To Be Discontinued` status is **not automatically equivalent to an active shortage**. NDCs extracted from free-text presentations are candidate identifiers until validated.

This notebook is **read-only**: it does not modify MySQL, your training labels, or your existing notebooks. It exports only summary tables and charts into `outputs/data_quality/`.

In [ ]:
from pathlib import Path
import csv, json, re, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 25)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# Works when run from either the project root or its Notebooks folder.
working = Path.cwd().resolve()
roots = [working, working.parent, working.parent.parent]
possible_data_dirs = []
for base in roots:
    possible_data_dirs.extend([base/'data', base/'Data', base/'raw', base/'Raw_Data', base])
# Keep order, remove duplicates.
possible_data_dirs = list(dict.fromkeys(possible_data_dirs))

def locate(name):
    for folder in possible_data_dirs:
        path = folder/name
        if path.is_file():
            return path
    raise FileNotFoundError(
        f'Could not find {name}. Place it in the project data/ folder. Searched: '
        + ', '.join(str(x) for x in possible_data_dirs)
    )

archive_path = locate('FDA_shortages_2023_archived.csv')
current_path = locate('drug-shortages-0001-of-0001.json.zip')
project_root = archive_path.parent.parent if archive_path.parent.name.lower()=='data' else working
output_dir = project_root/'outputs'/'data_quality'
output_dir.mkdir(parents=True, exist_ok=True)
print('Archived snapshot:', archive_path)
print('Later openFDA download:', current_path)
print('Output folder:', output_dir)

## 1 | Source inventory and provenance

**Archive:** `FDA_shortages_2023_archived.csv`, downloaded from an archived FDA page dated **December 21, 2023**. The date is based on the archived page, not inferred from individual posting dates.

**Later download:** `drug-shortages-0001-of-0001.json.zip`, an openFDA source downloaded in 2026. Its `meta.last_updated` field is printed below.

These are **different collection dates**, so a count comparison is descriptive—not a definitive count of shortages that occurred in a given year.

In [ ]:
with zipfile.ZipFile(current_path) as z:
    members = [m for m in z.namelist() if m.lower().endswith('.json')]
    if len(members) != 1:
        raise ValueError(f'Expected one JSON member; found {members}')
    with z.open(members[0]) as f:
        payload = json.load(f)
latest_raw = pd.DataFrame(payload['results'])
metadata = payload.get('meta', {})
print('openFDA metadata last_updated:', metadata.get('last_updated', 'not provided'))

# skip_blank_lines=True handles the empty first line in this archived CSV.
archive_raw = pd.read_csv(archive_path, encoding='utf-8-sig', skip_blank_lines=True, dtype=str, low_memory=False)
archive_raw.columns = [str(c).strip() for c in archive_raw.columns]

inventory = pd.DataFrame([
    {'source':'FDA archived CSV','capture':'2023-12-21 (archived page)','rows':len(archive_raw),'columns':len(archive_raw.columns)},
    {'source':'openFDA shortage JSON','capture':str(metadata.get('last_updated','2026 download')),'rows':len(latest_raw),'columns':len(latest_raw.columns)}
])
display(inventory)
inventory.to_csv(output_dir/'source_inventory.csv',index=False)
print('Archive fields:', ', '.join(archive_raw.columns))
print('openFDA fields:', ', '.join(latest_raw.columns))

## 2 | Standardize key fields without changing raw data

- **Posting year** is derived from the `Initial Posting Date`, not the date of a later status update.
- **NDCs in openFDA** come from `package_ndc`.
- **Archive candidate NDCs** are parsed from presentation text (which may include zero or multiple NDCs). This is a quality diagnostic, **not a validated one-to-one join**.
- Status values reflect **the respective snapshot**, not necessarily the status when first posted.

In [ ]:
def normalize_ndc(value):
    if pd.isna(value):
        return None
    digits = re.sub(r'\D','',str(value))
    return digits if len(digits) in (10,11) else None

ndc_pattern = re.compile(r'\bNDC\s*[:#]?\s*([0-9]{4,5}-[0-9]{3,4}-[0-9]{1,2}|[0-9]{10,11})\b',re.I)

def extract_ndcs(text):
    if pd.isna(text):
        return []
    return list(dict.fromkeys(filter(None,(normalize_ndc(v) for v in ndc_pattern.findall(str(text))))))

archive = archive_raw.copy()
archive['posting_date'] = pd.to_datetime(archive['Initial Posting Date'], errors='coerce')
archive['posting_year'] = archive['posting_date'].dt.year
archive['status_clean'] = archive['Status'].fillna('Missing').str.strip()
archive['candidate_ndcs'] = archive['Presentation'].map(extract_ndcs)
archive['candidate_ndc_count'] = archive['candidate_ndcs'].map(len)
archive['candidate_ndc_first'] = archive['candidate_ndcs'].map(lambda x: x[0] if x else None)

latest = latest_raw.copy()
latest['posting_date'] = pd.to_datetime(latest['initial_posting_date'], errors='coerce')
latest['posting_year'] = latest['posting_date'].dt.year
latest['status_clean'] = latest['status'].fillna('Missing').astype(str).str.strip()
latest['ndc_normalized'] = latest['package_ndc'].map(normalize_ndc)

print('Archived rows:',len(archive),'| openFDA rows:',len(latest))
print('Archive presentation rows with >=1 candidate NDC:',int((archive.candidate_ndc_count>0).sum()))
print('Archive presentation rows with multiple candidate NDCs:',int((archive.candidate_ndc_count>1).sum()))

## 3 | Completeness and consistency checks

A blank NDC, an invalid date, or repeated package identifier is an **investigation flag**, not automatic evidence that the record is wrong. Multiple entries can legitimately reference one package.

In [ ]:
quality = pd.DataFrame([
    {
        'source':'2023 archived CSV',
        'rows':len(archive),
        'missing_or_invalid_posting_date':int(archive.posting_date.isna().sum()),
        'missing_status':int((archive.status_clean=='Missing').sum()),
        'missing_candidate_ndc':int((archive.candidate_ndc_count==0).sum()),
        'multi_ndc_presentations':int((archive.candidate_ndc_count>1).sum()),
        'repeated_ndc_and_posting_date':int(archive.dropna(subset=['candidate_ndc_first','posting_date']).duplicated(['candidate_ndc_first','posting_date'],keep=False).sum()),
    },
    {
        'source':'2026 openFDA JSON',
        'rows':len(latest),
        'missing_or_invalid_posting_date':int(latest.posting_date.isna().sum()),
        'missing_status':int((latest.status_clean=='Missing').sum()),
        'missing_candidate_ndc':int(latest.ndc_normalized.isna().sum()),
        'multi_ndc_presentations':pd.NA,
        'repeated_ndc_and_posting_date':int(latest.dropna(subset=['ndc_normalized','posting_date']).duplicated(['ndc_normalized','posting_date'],keep=False).sum()),
    }
])
display(quality)
quality.to_csv(output_dir/'source_quality_checks.csv',index=False)

## 4 | Historical coverage by initial posting year

**Read this carefully:** The 2023 archive has **no opportunity to include future 2024–2025 postings**. They are marked *not applicable*, rather than zero. The 2026 download reflects its own collection rules and later availability of records.

In [ ]:
archive_counts = archive.groupby('posting_year').size()
latest_counts = latest.groupby('posting_year').size()
coverage = pd.DataFrame({'year':[2022,2023,2024,2025]})
coverage['archive_dec_2023'] = coverage.year.map(archive_counts)
coverage.loc[coverage.year>2023,'archive_dec_2023'] = np.nan
coverage['openfda_sep_2026'] = coverage.year.map(latest_counts).fillna(0).astype(int)
display(coverage)
coverage.to_csv(output_dir/'historical_coverage.csv',index=False)

fig,ax=plt.subplots(figsize=(9,4.3))
idx=np.arange(4)
ax.bar(idx-.18,coverage.archive_dec_2023,width=.36,label='Archived Dec 2023',color='#0d9488')
ax.bar(idx+.18,coverage.openfda_sep_2026,width=.36,label='openFDA Sep 2026',color='#24446a')
ax.set_xticks(idx,coverage.year.astype(str));ax.set_ylabel('Rows by initial posting year')
ax.set_title('FDA shortage source coverage differs across snapshots',fontweight='bold',loc='left')
ax.legend(frameon=False)
ax.text(.01,-.24,'Archive does not cover future years; source counts are not interchangeable shortage incidence.',transform=ax.transAxes,fontsize=8,color='#475569')
fig.tight_layout()
fig.savefig(output_dir/'historical_coverage.png',bbox_inches='tight',dpi=170)
plt.show()

## 5 | Status distribution and discontinuation sensitivity

This check helps identify potential **label-definition problems**. The `To Be Discontinued` category should be reviewed separately from `Current` shortage listings. It is not appropriate to automatically count or exclude all discontinuations without a documented target definition.

In [ ]:
statuses = (latest[latest.posting_year.between(2022,2025)]
            .groupby(['posting_year','status_clean']).size().unstack(fill_value=0))
display(statuses)
statuses.to_csv(output_dir/'openfda_status_by_posting_year.csv')

fig,ax=plt.subplots(figsize=(9,4.3))
colors=['#24446a','#0d9488','#e2a247','#8d73aa','#9aa5b1']
statuses.plot(kind='bar',stacked=True,ax=ax,color=colors[:len(statuses.columns)],width=.68)
ax.set_title('Later-snapshot status mix by initial posting year',loc='left',fontweight='bold')
ax.set_xlabel('Initial posting year');ax.set_ylabel('Records in 2026 source')
ax.legend(title='Status (2026 snapshot)',frameon=False,bbox_to_anchor=(1.01,1),loc='upper left')
fig.tight_layout()
fig.savefig(output_dir/'status_mix.png',bbox_inches='tight',dpi=170)
plt.show()

archive_2023 = archive.loc[archive.posting_year==2023,'status_clean'].value_counts().rename('archived_2023_snapshot')
latest_2023 = latest.loc[latest.posting_year==2023,'status_clean'].value_counts().rename('later_2026_snapshot')
status_comparison=pd.concat([archive_2023,latest_2023],axis=1).fillna(0).astype(int)
display(status_comparison)
status_comparison.to_csv(output_dir/'2023_status_snapshot_comparison.csv')

## 6 | Candidate NDC overlap for 2023

NDC overlap can show how much the two snapshots have in common. It **cannot by itself** establish which record is correct, whether a shortage resolved, or whether every archive record should become a training positive. Presentation text may mention a different NDC or multiple NDCs; NDC normalization does not fix all 10-digit versus 11-digit formatting ambiguity.

In [ ]:
arch_2023_ndcs = set(n for ns in archive.loc[archive.posting_year==2023,'candidate_ndcs'] for n in ns)
late_2023_ndcs = set(latest.loc[latest.posting_year==2023,'ndc_normalized'].dropna())
ndc_overlap = pd.DataFrame([
    {'metric':'2023 archive candidate NDCs','count':len(arch_2023_ndcs)},
    {'metric':'2023 openFDA source NDCs','count':len(late_2023_ndcs)},
    {'metric':'NDCs in both snapshots','count':len(arch_2023_ndcs & late_2023_ndcs)},
    {'metric':'Archive-only candidate NDCs','count':len(arch_2023_ndcs-late_2023_ndcs)},
    {'metric':'Later-source-only NDCs','count':len(late_2023_ndcs-arch_2023_ndcs)},
])
display(ndc_overlap)
ndc_overlap.to_csv(output_dir/'ndc_overlap_2023.csv',index=False)

## 7 | Optional: audit the actual PharmaOpt package-year panel

If `pharmaopt_integrated_package_year_v05.csv` is present in your project's `outputs/` directory, this section audits its **observed target prevalence by year**. It reads in chunks to avoid loading the entire ~328 MB file into memory.

If that file is not present, this step skips safely. This notebook does **not** alter the panel or train a model.

In [ ]:
panel_name='pharmaopt_integrated_package_year_v05.csv'
panel_locations=[project_root/'outputs'/panel_name, project_root/panel_name,project_root/'data'/panel_name]
panel_path=next((p for p in panel_locations if p.is_file()),None)
if panel_path is None:
    print('OPTIONAL PANEL AUDIT SKIPPED: panel CSV not found. Source-only audit is complete.')
else:
    header=pd.read_csv(panel_path,nrows=0).columns.tolist()
    print('Panel found:',panel_path)
    print('Column preview:',header[:20])
    year_col=next((x for x in ['prediction_year','year','cohort_year'] if x in header),None)
    label_col=next((x for x in ['shortage_target','target','label'] if x in header),None)
    if not year_col or not label_col:
        print('Panel audit skipped: could not identify year/target columns. Please share these column names.')
    else:
        aggregates={}
        for chunk in pd.read_csv(panel_path,usecols=[year_col,label_col],chunksize=100_000):
            chunk[year_col]=pd.to_numeric(chunk[year_col],errors='coerce')
            chunk[label_col]=pd.to_numeric(chunk[label_col],errors='coerce')
            for yr,g in chunk.groupby(year_col,dropna=False):
                a=aggregates.setdefault(yr,{'rows':0,'positives':0,'null_targets':0})
                a['rows']+=len(g)
                a['positives']+=int((g[label_col]==1).sum())
                a['null_targets']+=int(g[label_col].isna().sum())
        panel_audit=pd.DataFrame([{'prediction_year':yr,**values} for yr,values in aggregates.items()]).sort_values('prediction_year')
        panel_audit['positive_rate_pct']=100*panel_audit.positives/panel_audit.rows
        display(panel_audit)
        panel_audit.to_csv(output_dir/'panel_target_prevalence.csv',index=False)
        print('CAUTION: This describes existing labels; it does not validate their correctness.')

## 8 | Interpretation, decisions, and next research questions

### Findings to discuss with Professor Tushe or ASHP
- **Source provenance:** The two FDA snapshots were captured at different dates, so they are not interchangeable longitudinal datasets.
- **Temporal completeness:** A later snapshot can differ materially from an older archive. Historical label construction needs a documented source and inclusion rule.
- **Target ambiguity:** A `To Be Discontinued` listing and an active shortage may represent different outcomes; their treatment needs a clear business definition.
- **NDC granularity:** NDCs parsed from presentation text require validation against the FDA package directory before joining.
- **Historical NDC cohort:** A present-day NDC directory may not fully represent which packages were active in each historical year.
- **Evaluation:** Existing 2025 model metrics are development-holdout results, not independently validated deployment performance.

### Next data questions for ASHP
1. Can historical **record-level snapshots** or event histories for 2022–2025 be obtained?
2. Are **NDC/package identifiers** provided, and how are multiple package presentations handled?
3. Are **initial posting, resolution, and status-change dates** retained?
4. What is the **definition of shortage** in ASHP compared with FDA, and what use restrictions apply?

### Decision
**Do not rewrite training labels or retrain the model based only on these descriptive comparisons.** Preserve the current PharmaOpt pipeline, document the label-definition decision, and assess whether ASHP or another source can support a defensible event history.

The exported CSV tables and PNG charts in `outputs/data_quality/` can be used as technical exhibits when explaining the project.